In [38]:
import pandas as pd
import requests
import json
import numpy as np

url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"

r = requests.get(
    url,
    params={
        "format": "json",
        "date": 2023,
        "per_page": 300
    }
)

r.raise_for_status()

data_gdp = json.loads(r.content.decode("utf-8-sig"))

meta_gdp = data_gdp[0]
records_gdp = data_gdp[1]

df_gdp = pd.DataFrame(records_gdp)

df_gdp.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,1571.132704,,,1
1,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2023,1846.246811,,,1
2,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': '1A', 'value': 'Arab World'}",ARB,2023,7504.356377,,,1
3,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'S3', 'value': 'Caribbean small states'}",CSS,2023,17609.682707,,,1
4,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'B8', 'value': 'Central Europe and the ...",CEB,2023,22693.950447,,,1


In [39]:
df_gdp.columns

Index(['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit',
       'obs_status', 'decimal'],
      dtype='object')

In [40]:
df_gdp = df_gdp[["countryiso3code", "country", "date", "value", "indicator"]]

df_gdp.head()

,countryiso3code,country,date,value,indicator
0,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2023,1571.132704,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
1,AFW,"{'id': 'ZI', 'value': 'Africa Western and Cent...",2023,1846.246811,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
2,ARB,"{'id': '1A', 'value': 'Arab World'}",2023,7504.356377,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
3,CSS,"{'id': 'S3', 'value': 'Caribbean small states'}",2023,17609.682707,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
4,CEB,"{'id': 'B8', 'value': 'Central Europe and the ...",2023,22693.950447,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."


In [41]:
df_gdp = df_gdp.rename(columns = {
    "countryiso3code" : "iso3",
    "country" : "Country",
    "date" : "year",
    "value" : "gdp_per_capita"
})

df_gdp.head()

,iso3,Country,year,gdp_per_capita,indicator
0,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2023,1571.132704,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
1,AFW,"{'id': 'ZI', 'value': 'Africa Western and Cent...",2023,1846.246811,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
2,ARB,"{'id': '1A', 'value': 'Arab World'}",2023,7504.356377,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
3,CSS,"{'id': 'S3', 'value': 'Caribbean small states'}",2023,17609.682707,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."
4,CEB,"{'id': 'B8', 'value': 'Central Europe and the ...",2023,22693.950447,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap..."


In [42]:
df_gdp["Country"] = df_gdp["Country"].apply(
    lambda x: x["value"] if isinstance(x, dict) else x
)

df_gdp["indicator"] = df_gdp["indicator"].apply(
    lambda x: x["value"] if isinstance(x, dict) else x
)

In [43]:
url = "https://api.worldbank.org/v2/country"

r = requests.get(url,params={"format": "json","per_page": 400})

r.raise_for_status()
print(r.status_code)
countries_data = json.loads(r.content.decode("utf-8-sig"))
countries_records = countries_data[1]

df_countries = pd.DataFrame(countries_records)

200


In [44]:
valid_countries = df_countries[
    df_countries["region"].apply(lambda x: x["id"] != "NA")
]

valid_iso3 = set(valid_countries["id"])

df_gdp = df_gdp[
    df_gdp["iso3"].isin(valid_iso3)
].copy()

In [45]:
df_gdp = df_gdp[["iso3", "Country", "year", "gdp_per_capita"]]

In [46]:
df_gdp.head()

,iso3,Country,year,gdp_per_capita
49,AFG,Afghanistan,2023,413.757895
50,ALB,Albania,2023,9740.702341
51,DZA,Algeria,2023,5370.477235
52,ASM,American Samoa,2023,NaN
53,AND,Andorra,2023,46812.448450


In [47]:
df_gdp.shape

(217, 4)

In [48]:
def get_indicator(code, column_name):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}"

    r = requests.get(url, params={"format": "json","date": 2023,"per_page": 300})

    r.raise_for_status()

    data = json.loads(r.content.decode("utf-8-sig"))
    records = data[1]

    df = pd.DataFrame(records)

   
    df = df[df["countryiso3code"].isin(valid_iso3)].copy()

    df["Country"] = df["country"].apply(
        lambda x: x["value"] if isinstance(x, dict) else x
    )

    df = df.rename(columns={
        "countryiso3code": "iso3",
        "date": "year",
        "value": column_name
    })

    df[column_name] = pd.to_numeric(
        df[column_name],
        errors="coerce"
    )

    return df[
        ["iso3", "Country", "year", column_name]
    ]

In [49]:
df_life = get_indicator(
    "SP.DYN.LE00.IN",
    "life_expectancy"
)

df_life.head()

,iso3,Country,year,life_expectancy
49,AFG,Afghanistan,2023,66.035
50,ALB,Albania,2023,79.602
51,DZA,Algeria,2023,76.261
52,ASM,American Samoa,2023,72.852
53,AND,Andorra,2023,84.041


In [50]:
df_unemployment = get_indicator(
    "SL.UEM.TOTL.ZS",
    "unemployment"
)

df_unemployment.head()

,iso3,Country,year,unemployment
49,AFG,Afghanistan,2023,14.008
50,ALB,Albania,2023,10.669
51,DZA,Algeria,2023,11.645
52,ASM,American Samoa,2023,NaN
53,AND,Andorra,2023,NaN


In [51]:
df_population = get_indicator(
    "SP.POP.TOTL",
    "population"
)

df_population.head()

,iso3,Country,year,population
49,AFG,Afghanistan,2023,41454761.0
50,ALB,Albania,2023,2411658.0
51,DZA,Algeria,2023,46164219.0
52,ASM,American Samoa,2023,47521.0
53,AND,Andorra,2023,80856.0


In [52]:
df_economic = df_gdp.merge(
    df_life[["iso3", "life_expectancy"]],
    on="iso3",
    how="left",
    validate="one_to_one"
)

df_economic = df_economic.merge(
    df_unemployment[["iso3", "unemployment"]],
    on="iso3",
    how="left",
    validate="one_to_one"
)

df_economic = df_economic.merge(
    df_population[["iso3", "population"]],
    on="iso3",
    how="left",
    validate="one_to_one"
)

df_economic.head()

,iso3,Country,year,gdp_per_capita,life_expectancy,unemployment,population
0,AFG,Afghanistan,2023,413.757895,66.035,14.008,41454761.0
1,ALB,Albania,2023,9740.702341,79.602,10.669,2411658.0
2,DZA,Algeria,2023,5370.477235,76.261,11.645,46164219.0
3,ASM,American Samoa,2023,NaN,72.852,NaN,47521.0
4,AND,Andorra,2023,46812.448450,84.041,NaN,80856.0


In [53]:
df_economic.shape

(217, 7)

In [54]:
df_economic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   iso3             217 non-null    object 
 1   Country          217 non-null    object 
 2   year             217 non-null    object 
 3   gdp_per_capita   204 non-null    float64
 4   life_expectancy  217 non-null    float64
 5   unemployment     184 non-null    float64
 6   population       217 non-null    float64
dtypes: float64(4), object(3)
memory usage: 12.0+ KB


In [55]:
df_economic.isna().sum()

iso3                0
Country             0
year                0
gdp_per_capita     13
life_expectancy     0
unemployment       33
population          0
dtype: int64

In [56]:
df_economic[["gdp_per_capita","life_expectancy","unemployment","population"]].describe()

,gdp_per_capita,life_expectancy,unemployment,population
count,204.000000,217.000000,184.000000,2.170000e+02
mean,22295.539426,73.790943,6.778283,3.704862e+07
std,33559.351447,7.122870,5.520379,1.421754e+08
min,249.795164,54.462000,0.130000,9.816000e+03
25%,2945.926786,68.484000,3.163500,8.263530e+05
50%,7862.027081,74.552000,5.210500,6.623183e+06
75%,30638.403469,79.594000,8.427500,2.641820e+07
max,256799.911212,86.372000,34.610000,1.438070e+09


In [57]:
correlation_matrix = df_economic[["gdp_per_capita","life_expectancy","unemployment"]].corr()

correlation_matrix

,gdp_per_capita,life_expectancy,unemployment
gdp_per_capita,1.000000,0.620409,-0.182598
life_expectancy,0.620409,1.000000,-0.150432
unemployment,-0.182598,-0.150432,1.000000


In [58]:

df_economic["log_gdp"] = np.log10( df_economic["gdp_per_capita"])

In [59]:
df_economic[["log_gdp","life_expectancy","unemployment"]].corr()

,log_gdp,life_expectancy,unemployment
log_gdp,1.000000,0.855143,-0.073837
life_expectancy,0.855143,1.000000,-0.150432
unemployment,-0.073837,-0.150432,1.000000


In [60]:
df_economic.to_csv("world_bank_economic_2023.csv",index=False)

In [61]:
df_economic.head()

,iso3,Country,year,gdp_per_capita,life_expectancy,unemployment,population,log_gdp
0,AFG,Afghanistan,2023,413.757895,66.035,14.008,41454761.0,2.616746
1,ALB,Albania,2023,9740.702341,79.602,10.669,2411658.0,3.988590
2,DZA,Algeria,2023,5370.477235,76.261,11.645,46164219.0,3.730013
3,ASM,American Samoa,2023,NaN,72.852,NaN,47521.0,NaN
4,AND,Andorra,2023,46812.448450,84.041,NaN,80856.0,4.670361
